## Pregunta
¿Cómo varían los montos de transacción entre canales?

## Fuente de datos
Dataset sintético generado con numpy (seed 42), 200 filas.

## Unidades
Unidades sintéticas (synthetic units); no representan una moneda real.

## Política de datos faltantes
Se excluye la fila con monto faltante antes de calcular estadísticas.

In [0]:
%pip install plotly
dbutils.library.restartPython()

In [0]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px

rng = np.random.default_rng(42)
df = pd.DataFrame({
    "channel": ["Online"] * 100 + ["Store"] * 100,
    "amount": np.round(rng.lognormal(3.5, 0.6, 200), 2),
})
df.loc[0, "amount"] = np.nan

print("Versions:", np.__version__, pd.__version__, plotly.__version__)
print("Rows:", len(df))
print("Missing values:", df[["amount", "channel"]].isna().sum())

clean = df.dropna(subset=["amount", "channel"]).copy()
if len(clean) < 2 or not np.isfinite(clean["amount"]).all():
    raise ValueError("Need at least two finite amounts")

print(clean["amount"].describe().round(2))
print("Median:", round(clean["amount"].median(), 2))
print(clean.groupby("channel")["amount"].agg(
    ["count", "mean", "median"]
).round(2))

In [0]:
histogram = px.histogram(
    clean, x="amount", nbins=20, title="Transaction amount distribution",
    labels={"amount": "Amount (synthetic units)"},
)
box = px.box(
    clean, x="channel", y="amount", title="Amounts by channel",
    labels={"amount": "Amount (synthetic units)", "channel": "Channel"},
)
histogram.show()
box.show()

## Interpretación

**Observación con estadística:** El promedio de los montos es de 37.61 y la mediana de 32.06. Esto significa que hay algunos montos bastante altos que hacen que el promedio suba. También se puede ver que los datos están algo dispersos, ya que la desviación estándar es de 23.12.

**Observación con gráfica:** En el histograma se observa que la mayoría de los montos están entre 10 y 50, aunque hay algunos mucho más altos, llegando hasta 190.25. En el box plot, Online (32.78) y Store (30.64) tienen valores muy parecidos en sus medianas. Sin embargo, Store tiene más valores atípicos y una mayor extensión en la parte superior.

**Grupos desiguales:** Se tomaron 99 registros de Online y 100 de Store, porque uno de los registros de Online no tenía monto y se tuvo que quitar del análisis.

## Limitación

Los datos son sintéticos y ambos canales parten de la misma distribución, por lo que las diferencias que vemos pueden ser simplemente por el azar y no porque realmente exista una diferencia entre Online y Store. Además, al quitar el registro que no tenía monto, los resultados de Online pudieron cambiar un poco.